In [ ]:
'''
CODE USED TO GET WEIGHTS JUST FROM SPECTRA!

weights = np.load("${repo_root}/assets/lss/dirichlet_weights.npy")

print(weights.shape)


def read_lss(lss_path):
    """
    Load a numpy array from the given lss file path and convert it to a pandas DataFrame.
    """
    df = pd.read_csv(lss_path, sep="\t", header=0)
    return df

just_led_illuminations = read_lss("${repo_root}/assets/lss/just_led.lss")
#remove the first column
just_led_illuminations = just_led_illuminations.iloc[:, 1:]
start_wavelength = 300
end_wavelength = 1000
step = 5
start_idx = int((380 - start_wavelength) / step)  # (380 - 300) / 5 = 16
end_idx = int((730 - start_wavelength) / step)    # (730 - 300) / 5 = 86
just_led_illuminations = just_led_illuminations.iloc[:, start_idx:end_idx+1]  # +1 because end is exclusive
just_led_illuminations = just_led_illuminations.to_numpy() #Illuminations * 71

all_illuminations = read_lss("${repo_root}/assets/lss/combined.lss")
all_illuminations = all_illuminations.iloc[:, 1:]
all_illuminations = all_illuminations.iloc[:, start_idx:end_idx+1]
all_illuminations = all_illuminations.to_numpy() #Illuminations *

print(just_led_illuminations.shape)
print(all_illuminations.shape)

# Right pseudoinverse of J (shape: 71x7)
just_led_illuminations_p_inv = np.linalg.pinv(just_led_illuminations)

# Weight matrix (shape: 834x7)
weights = all_illuminations @ just_led_illuminations_p_inv


# A: (834, 71), W: (834, 7), J: (7, 71)
A_hat = weights @ just_led_illuminations   # reconstructed illuminations (834, 71)

# Row-wise L2 error
row_errors = np.linalg.norm(all_illuminations - A_hat, axis=1)

# If you want relative error (normalized by original row magnitude):
row_rel_errors = row_errors / np.linalg.norm(all_illuminations, axis=1)

print("Absolute error per row:", row_errors)
print("Relative error per row:", row_rel_errors)

'''

In [ ]:
from pathlib import Path
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
cameras = ["Sony", "Canon", "Pixel", "Samsung"]
for camera in cameras:
    captures_dir = Path("${data_root}/LightboxCaptures/colorchecker24")/camera
    input_dir = captures_dir/"patches"
    output_dir = captures_dir/"blended_patches"
    os.makedirs(output_dir, exist_ok=True)

    weights = np.load("illumination_blending_weights.npy")
    all_npy_files = sorted(os.listdir(input_dir))

    #read first 7 npy files and stack them into a numpy array
    led_npy_files = all_npy_files[:7]
    led_ims = [np.load(os.path.join(input_dir, f), allow_pickle=True).item()["patches"] for f in led_npy_files]
    led_ims = np.stack(led_ims, axis=0)  # Shape: (7, H, W, C)

    #visualize the 7 images at 1/16 scale

    scale_viz = 10
    plt.figure(figsize=(15, 10))
    for i in range(7):
        plt.subplot(2, 4, i+1)
        print("Max", np.max(led_ims[i]))
        plt.imshow(led_ims[i]*scale_viz) 
        plt.axis('off')
        plt.title(f'LED {i+1}')
    plt.tight_layout()

    #blend the images using the weights to generate all 834 images\
    blended_ims = np.tensordot(weights, led_ims, axes=([1], [0]))  # Shape: (834, H, W, C)
    print("Blended ims shape:", blended_ims.shape)  # Should print (834, H, W, C)

    #write the bleded images to Sony/simulated_patches
    for i in range(blended_ims.shape[0]):
        input_file_name = all_npy_files[i] 
        input_npy = np.load(os.path.join(input_dir, input_file_name), allow_pickle=True).item()
        patches = blended_ims[i]
        whitepoint = patches[-1, 0, :]
        output_npy = {"blended_patches": patches, "blended_whitepoint": whitepoint}
        output_npy |= {k: v for k, v in input_npy.items() if k not in output_npy} #add other keys from input_npy to output_npy if they don't exist in output_npy already
        output_path = os.path.join(output_dir, input_file_name)    
        np.save(output_path, output_npy)

    #visualize the first 10 blended images at 1/16 scale and the corresponding ground truth images
    gt_npy_files = sorted(os.listdir(input_dir))
    gt_ims = [np.load(os.path.join(input_dir, f), allow_pickle=True).item()["patches"] for f in gt_npy_files]
    gt_ims = np.stack(gt_ims, axis=0)  # Shape: (10, H, W, C)
    print("GT ims shape:", gt_ims.shape)  # Should print (10, H, W, C)

    plt.figure(figsize=(20, 10))
    start = 700
    end = 709
    for i in range(start, end):
        plt.subplot(2, (end-start), i-start+1)
        plt.imshow(blended_ims[i]*scale_viz)  # Assuming the images are in the range [0, 255]
        plt.axis('off')
        plt.title(f'Blended {i+1}')
        
        plt.subplot(2, (end-start), i-start+1+(end-start))
        plt.imshow(gt_ims[i]*scale_viz)  # Assuming the images are in the range [0, 255]
        plt.axis('off')
        plt.title(f'GT {i+1}')


    #Compute the mean squared error between the blended images and the ground truth images per image
    mse_per_image = np.mean((blended_ims - gt_ims) ** 2, axis=(1, 2, 3))
    rmse_per_image = np.sqrt(mse_per_image)

    #PSNR per image
    psnr_per_image = 20 * np.log10(1.0 / np.sqrt(mse_per_image))
    print("PSNR per image:", np.mean(psnr_per_image))
    print("Mean RMSE per image:", np.mean(rmse_per_image))
